# 01. Pozyskanie i opis danych

Ten notebook odpowiada etapowi: opis źródła danych, liczby obserwacji, struktury zbioru i potencjalnych problemów jakościowych.
Nie zapisuje zmian do bazy wejściowej.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "database" / "NajnowszaWersjaBazy1205.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH exists:", DATA_PATH.exists())


PROJECT_ROOT: C:\Users\szymon\projekt_reddit
DATA_PATH exists: True


In [2]:
df = pd.read_csv(DATA_PATH)
df["TIMESTAMP"] = pd.to_datetime(df["TIMESTAMP"])

print("Wymiary bazy:", df.shape)
print("Zakres czasu:", df["TIMESTAMP"].min(), "->", df["TIMESTAMP"].max())
print("Liczba unikalnych par SOURCE -> TARGET:", df[["SOURCE_SUBREDDIT", "TARGET_SUBREDDIT"]].drop_duplicates().shape[0])


Wymiary bazy: (49918, 95)
Zakres czasu: 2013-12-31 16:39:58 -> 2017-04-19 00:15:44
Liczba unikalnych par SOURCE -> TARGET: 29621


In [3]:
sentiment_counts = pd.DataFrame({
    "LINK_SENTIMENT": df["LINK_SENTIMENT"].value_counts().sort_index(),
    "Content_Sentiment": df["Content_Sentiment"].value_counts().sort_index(),
})
display(sentiment_counts)


,LINK_SENTIMENT,Content_Sentiment
-1,3854.0,7868
0,NaN,34183
1,46064.0,7867


In [4]:
key_columns = [
    "POST_ID", "TIMESTAMP", "SOURCE_SUBREDDIT", "TARGET_SUBREDDIT",
    "Raw_Title", "Raw_Content", "LINK_SENTIMENT", "Content_Sentiment",
    "LIWC_Anger", "Average word length", "LIWC_Conj", "Automated readability index",
]
missing = df[key_columns].isna().sum().to_frame("missing_values")
display(missing)


,missing_values
POST_ID,0
TIMESTAMP,0
SOURCE_SUBREDDIT,0
TARGET_SUBREDDIT,0
Raw_Title,0
Raw_Content,0
LINK_SENTIMENT,0
Content_Sentiment,0
LIWC_Anger,0
Average word length,0


In [5]:
numeric_cols = [
    "LIWC_Anger", "Average word length", "LIWC_Conj",
    "Automated readability index", "LIWC_CogMech", "Number of words"
]
display(df[numeric_cols].describe().T)


,count,mean,std,min,25%,50%,75%,max
LIWC_Anger,49918.0,0.004877,0.010288,0.000000,0.000000,0.000000,0.005882,0.245902
Average word length,49918.0,5.332482,1.412862,1.925105,4.558932,4.954275,5.656250,175.950000
LIWC_Conj,49918.0,0.040841,0.028075,0.000000,0.018692,0.043478,0.061404,0.267760
Automated readability index,49918.0,21.480778,15.482949,5.538291,15.110105,18.557134,24.298220,822.711000
LIWC_CogMech,49918.0,0.116731,0.062694,0.000000,0.075000,0.125326,0.161736,0.363636
Number of words,49918.0,232.330422,354.172119,9.000000,42.000000,107.000000,263.000000,7094.000000


## Wniosek po etapie

Baza ma gotową strukturę do modelowania: identyfikatory postów, czas, pary subredditów, tekst, sentyment linku oraz cechy LIWC i właściwości tekstu.
Zgodnie z założeniem projektu nie wykonujemy czyszczenia danych, tylko sprawdzamy ich strukturę i kompletność w kolumnach używanych dalej.
